# 02. `SemanticEmbedder` 생명주기 모의 실습

목표: Chrome Embedding API의 `availability() -> create() -> embed() -> destroy()` 흐름을 Python 모의 객체로 익힌다.

실행 방법:
1. 이 노트북을 위에서 아래로 실행한다.
2. 브라우저 API 자체는 `semantic_embedder_demo.html`에서 테스트한다.
3. 이 노트북은 Python 표준 라이브러리만 사용한다.

이 실습은 실제 모델 품질을 재현하지 않는다. API 사용 순서와 오류 처리를 학습하기 위한 축소 구현이다.

In [ ]:
import hashlib
import math


def cosine_similarity(vec_a, vec_b):
    if len(vec_a) != len(vec_b):
        raise ValueError("Vectors must have the same dimension")
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

## 1. 모의 embedder 구현

아래 클래스는 실제 Chrome API와 비슷한 메서드 이름을 갖는다. `availability()`가 `available`일 때만 `create()`가 성공하고, `destroy()` 이후에는 `embed()`를 호출할 수 없게 만든다.

In [ ]:
class MockSemanticEmbedder:
    _availability = "available"
    _dimension = 12
    _space_id = "mock-embedding-space-v1"

    @classmethod
    def availability(cls):
        """실제 API의 SemanticEmbedder.availability()에 해당한다."""
        return cls._availability

    @classmethod
    def create(cls):
        """사용 가능 상태일 때만 embedder 인스턴스를 생성한다."""
        if cls.availability() != "available":
            raise RuntimeError("Embedding model is not ready yet")
        return cls()

    def __init__(self):
        self.destroyed = False

    def embed(self, inputs, options=None):
        """문자열 또는 문자열 배열을 벡터 배열로 바꾼다."""
        if self.destroyed:
            raise RuntimeError("Cannot use an embedder after destroy()")

        options = options or {}
        task_type = options.get("taskType", "raw")
        if isinstance(inputs, str):
            batch = [inputs]
        else:
            batch = list(inputs)

        embeddings = []
        for text in batch:
            embeddings.append({
                "values": self._hash_embedding(text, task_type),
                "taskType": task_type,
                "spaceId": self._space_id,
            })
        return {"embeddings": embeddings}

    def destroy(self):
        """실제 API의 리소스 해제 습관을 학습하기 위한 메서드다."""
        self.destroyed = True

    @classmethod
    def _hash_embedding(cls, text, task_type):
        """결정적인 장난감 벡터를 만든다.

        taskType을 해시에 포함해 같은 텍스트라도 태스크가 다르면 다른 공간처럼 보이게 한다.
        실제 API에서도 용도에 맞는 taskType을 섞어 쓰면 비교 정책을 조심해야 한다.
        """
        seed = f"{task_type}\n{text}".encode("utf-8")
        digest = hashlib.sha256(seed).digest()
        raw = [(digest[i] / 255.0) * 2 - 1 for i in range(cls._dimension)]
        norm = math.sqrt(sum(value * value for value in raw)) or 1.0
        return [value / norm for value in raw]

## 2. 사용 가능 여부 확인 후 생성하기

원문은 모델 다운로드가 끝나기 전 `create()`가 실패할 수 있다고 설명한다. 그래서 생성 전에 `availability()`를 확인하는 흐름을 습관화해야 한다.

In [ ]:
if MockSemanticEmbedder.availability() == "available":
    embedder = MockSemanticEmbedder.create()
    print("embedder created")
else:
    print("model is not ready")

## 3. 단일 문자열과 배치 입력

원문 API는 단일 문자열과 문자열 배열을 모두 지원한다. 반환값은 항상 `embeddings` 배열을 포함한다고 생각하면 다루기 쉽다.

In [ ]:
single = embedder.embed("Built-in AI runs on the user's device", {"taskType": "semantic-similarity"})
print("single count:", len(single["embeddings"]))
print("dimension:", len(single["embeddings"][0]["values"]))

batch = embedder.embed([
    "On-device embeddings improve privacy",
    "Server APIs can add network latency",
], {"taskType": "semantic-similarity"})
print("batch count:", len(batch["embeddings"]))

## 4. `taskType` 차이 확인

같은 텍스트라도 `semantic-similarity`, `retrieval-query`, `retrieval-document`는 목적이 다르다. 실제 모델에서는 같은 공간 관리가 더 복잡할 수 있으므로, 인덱스 저장 시 어떤 태스크로 만들었는지 기록해야 한다.

In [ ]:
text = "Find local notes about Chrome AI"

similarity_vec = embedder.embed(text, {"taskType": "semantic-similarity"})["embeddings"][0]["values"]
query_vec = embedder.embed(text, {"taskType": "retrieval-query"})["embeddings"][0]["values"]
document_vec = embedder.embed(text, {"taskType": "retrieval-document"})["embeddings"][0]["values"]

print("same text, similarity vs query:", round(cosine_similarity(similarity_vec, query_vec), 3))
print("query vs document:", round(cosine_similarity(query_vec, document_vec), 3))

## 5. 리소스 해제

`destroy()`는 브라우저 리소스를 명시적으로 돌려주는 습관을 만든다. 실제 API에서는 모델 세션이나 메모리 자원을 오래 붙잡지 않도록 작업이 끝난 뒤 호출하는 편이 좋다.

In [ ]:
embedder.destroy()

try:
    embedder.embed("This should fail")
except RuntimeError as error:
    print("expected error:", error)

## 정리

- `availability()` 확인 없이 `create()`를 호출하면 초기 다운로드 상태에서 실패할 수 있다.
- 단일 입력과 배치 입력은 모두 `embeddings` 배열로 다룬다.
- `taskType`은 제품 기능의 의미를 나타내므로 저장 메타데이터에 포함해야 한다.
- 작업 후 `destroy()`로 리소스를 해제한다.